# Module 4 · HuggingFace Pipelines & Transfer Learning

**Track:** Natural Language Processing (0 to Masterclass)  
**Prerequisites:** NLP 05 (Pre-training & BERT)  
**Difficulty:** ⭐⭐⭐ Advanced  
**Estimated Time:** 120–150 minutes

---

## 🎓 Socratic Deep Check

> *"HuggingFace has 500K+ models on its Hub. You need a model for detecting toxic comments in production. How do you choose between fine-tuning a base model yourself vs. using an existing fine-tuned model? What are the risks of each approach?"*

---

## Why This Notebook Completes the NLP Track

NLP 01-05 taught you the THEORY: tokenization → embeddings → RNNs → attention → pre-training.

This notebook teaches the PRACTICE: how to use HuggingFace's ecosystem to solve real NLP tasks in production. Everything here directly connects to the main curriculum's NB11 (fine-tuning) and NB24-34 (LLMOps).


## 📑 Table of Contents

* [🎓 Socratic Deep Check](#socratic-deep-check)
* [Why This Notebook Completes the NLP Track](#why-this-notebook-completes-the-nlp-track)
  * [🎓 Junior to Senior: Concept Bridge](#junior-to-senior-concept-bridge)
* [1 · HuggingFace Hub: Finding & Evaluating Models](#1-huggingface-hub-finding-evaluating-models)
* [2 · Pipelines: Zero-Code NLP (Task Taxonomy)](#2-pipelines-zero-code-nlp-task-taxonomy)
* [3 · HuggingFace Datasets Deep Dive](#3-huggingface-datasets-deep-dive)
  * [🎓 Why This Matters](#why-this-matters)
  * [3.1 · Two Modes: Map-style vs Streaming](#31-two-modes-map-style-vs-streaming)
  * [3.2 · When to Use Streaming](#32-when-to-use-streaming)
* [4 · Domain-Adaptive Pretraining (DAPT)](#4-domain-adaptive-pretraining-dapt)
  * [🎓 The Production Problem](#the-production-problem)
  * [Domain Adaptation (Continued Pretraining)](#domain-adaptation-continued-pretraining)
* [5 · Fine-tuning with Trainer API + Evaluation Metrics](#5-fine-tuning-with-trainer-api-evaluation-metrics)
  * [🎓 Junior to Senior: The Gap in Most Tutorials](#junior-to-senior-the-gap-in-most-tutorials)
  * [The Complete Pipeline](#the-complete-pipeline)
  * [🎓 Loss vs. F1: The Deep Explanation](#loss-vs-f1-the-deep-explanation)
  * [🎓 Bridge to the Main Curriculum: What's Next?](#bridge-to-the-main-curriculum-whats-next)
  * [5.3 · Overcoming the GPU VRAM Bottleneck](#53-overcoming-the-gpu-vram-bottleneck)
    * [1. Gradient Accumulation](#1-gradient-accumulation)
    * [2. Automatic Mixed Precision (AMP)](#2-automatic-mixed-precision-amp)
    * [Engineering Pattern: Optimized TrainingArguments](#engineering-pattern-optimized-trainingarguments)
  * [5.1 · Advanced Technique: Handling Imbalanced Datasets](#51-advanced-technique-handling-imbalanced-datasets)
  * [5.2 · Advanced Technique: Knowledge Distillation](#52-advanced-technique-knowledge-distillation)
    * [How It Works:](#how-it-works)
  * [5.4 · Advanced Technique: Token Classification & Label Alignment](#54-advanced-technique-token-classification-label-alignment)
    * [The Production Nightmare: Misalignment](#the-production-nightmare-misalignment)
    * [The `-100` Rule](#the-100-rule)
    * [Python Utility: Realigning Labels](#python-utility-realigning-labels)
  * [5.5 · Handling Long Documents: The Sliding Window Strategy](#55-handling-long-documents-the-sliding-window-strategy)
    * [1. The 512-Token Hard Ceiling](#1-the-512-token-hard-ceiling)
    * [2. The Algorithm: Chunk, Infer, Pool](#2-the-algorithm-chunk-infer-pool)
  * [5.6 · Strategic Fine-Tuning: Layer Freezing & Discriminative Learning Rates](#56-strategic-fine-tuning-layer-freezing-discriminative-learning-rates)
    * [1. The Catastrophic Forgetting Risk](#1-the-catastrophic-forgetting-risk)
    * [2. Strategy A: Selective Layer Freezing](#2-strategy-a-selective-layer-freezing)
    * [3. Strategy B: Discriminative Fine-Tuning (ULMFiT)](#3-strategy-b-discriminative-fine-tuning-ulmfit)
* [6 · Production Deployment Patterns](#6-production-deployment-patterns)
  * [Connection to the Main Curriculum](#connection-to-the-main-curriculum)
  * [🎓 DEEP QUESTION ANSWERED](#deep-question-answered)
  * [6.1 · Optimized Inference: ONNX Export & Dynamic Quantization](#61-optimized-inference-onnx-export-dynamic-quantization)
    * [The Production Reality](#the-production-reality)
    * [Elite Pattern: Using HuggingFace `optimum`](#elite-pattern-using-huggingface-optimum)
* [✅ Knowledge Check](#knowledge-check)
  * [Q1: Pipeline internals](#q1-pipeline-internals)
  * [Q2: Learning rate](#q2-learning-rate)
  * [Q3: Zero-shot vs fine-tuning](#q3-zero-shot-vs-fine-tuning)
  * [Q4: Streaming datasets](#q4-streaming-datasets)
  * [Q5: Loss vs F1](#q5-loss-vs-f1)
* [🔨 Practical Practice](#practical-practice)
  * [Exercise 1: NER Pipeline for RAG](#exercise-1-ner-pipeline-for-rag)
  * [Exercise 2: Model Comparison](#exercise-2-model-comparison)
  * [Exercise 3: Streaming + Metrics](#exercise-3-streaming-metrics)
* [🎉 NLP Track Complete!](#nlp-track-complete)


### 🎓 Junior to Senior: Concept Bridge

**The Senior Perspective:** Juniors pick the highest-downloaded model. Seniors evaluate: license compatibility, benchmark scores on THEIR domain, model size vs. latency budget, last commit date, and whether the model was trained on data similar to their production data.

**Mental Model:** The HuggingFace Hub is like npm/pip for ML models. But unlike software packages, ML models have SILENT failures — a bad model won't crash, it'll just give wrong answers confidently. You must evaluate before deploying.

**Common Junior Pitfall:** Using `pipeline()` in production without controlling batch size, device placement, or model quantization. Default pipeline settings are for demos, not production throughput.

---

In [ ]:
!pip install -q transformers datasets evaluate accelerate scikit-learn

## 1 · HuggingFace Hub: Finding & Evaluating Models

Before using ANY model, check:

| Criterion | Why It Matters | Where to Check |
|-----------|---------------|----------------|
| **License** | Apache-2.0 = commercial OK, CC-BY-NC = non-commercial only | Model card |
| **Downloads** | Popularity ≈ community trust | Hub page |
| **Benchmarks** | Does it perform on YOUR task? | Model card / Papers With Code |
| **Model size** | Fits your GPU/latency budget? | Model card |
| **Last updated** | Stale models may have unpatched issues | Hub page |
| **Training data** | Was it trained on data similar to yours? | Model card |
| **Base model** | What was it fine-tuned from? | Model card |

---
## 2 · Pipelines: Zero-Code NLP (Task Taxonomy)

HuggingFace `pipeline()` wraps tokenization + model + post-processing into one call.

In [ ]:
from transformers import pipeline

# ============================
# TASK 1: Sentiment Analysis
# ============================
print('=== Sentiment Analysis ===')
classifier = pipeline('sentiment-analysis', model='distilbert-base-uncased-finetuned-sst-2-english')

reviews = [
    'This product exceeded all my expectations!',
    'Worst purchase I have ever made. Total garbage.',
    'It works fine, nothing extraordinary though.',
]
results = classifier(reviews)
for review, result in zip(reviews, results):
    print(f'  [{result["label"]:>8} {result["score"]:.2%}] "{review[:50]}"')

# ============================
# TASK 2: Named Entity Recognition (NER)
# ============================
print('\n=== Named Entity Recognition ===')
ner = pipeline('ner', model='dslim/bert-base-NER', aggregation_strategy='simple')

text = 'Elon Musk announced that Tesla will open a new factory in Berlin, Germany next year.'
entities = ner(text)
print(f'  Text: "{text}"')
for ent in entities:
    print(f'  {ent["entity_group"]:>5}: "{ent["word"]}" (confidence: {ent["score"]:.2%})')

In [ ]:
from transformers import pipeline

# ============================
# TASK 3: Question Answering (Extractive)
# ============================
print('=== Extractive Question Answering ===')
qa = pipeline('question-answering', model='distilbert-base-cased-distilled-squad')

context = """PyTorch was developed by Meta's AI Research lab (FAIR). It was first released 
in September 2016 and has since become the most popular deep learning framework for research.
As of 2024, it powers over 60% of academic publications in machine learning."""

questions = [
    'Who developed PyTorch?',
    'When was PyTorch first released?',
    'What percentage of academic publications use PyTorch?',
]

for q in questions:
    result = qa(question=q, context=context)
    print(f'  Q: {q}')
    print(f'  A: "{result["answer"]}" (confidence: {result["score"]:.2%})')
    print()

# ============================
# TASK 4: Summarization
# ============================
print('=== Summarization ===')
summarizer = pipeline('summarization', model='facebook/bart-large-cnn')

article = """The field of artificial intelligence has seen remarkable progress in recent years, 
particularly with the development of large language models. These models, trained on vast amounts 
of text data, have demonstrated capabilities in understanding and generating human language that 
were previously thought impossible. Companies like OpenAI, Google, and Meta have all released 
increasingly powerful models, each pushing the boundaries of what AI can achieve. However, this 
rapid progress has also raised concerns about safety, bias, and the potential misuse of these 
powerful tools. Researchers are now working on alignment techniques to ensure these models 
behave in ways that are helpful, honest, and harmless."""

summary = summarizer(article, max_length=60, min_length=20)
print(f'  Original: {len(article.split())} words')
print(f'  Summary:  "{summary[0]["summary_text"]}"')
print(f'  Compressed to: {len(summary[0]["summary_text"].split())} words')

In [ ]:
from transformers import pipeline

# ============================
# TASK 5: Zero-Shot Classification
# ============================
print('=== Zero-Shot Classification ===')
print('(No training data needed! Classify into ANY categories)')

zero_shot = pipeline('zero-shot-classification', model='facebook/bart-large-mnli')

text = 'The patient showed elevated troponin levels and ST-segment depression on the ECG.'
labels = ['cardiology', 'neurology', 'orthopedics', 'dermatology']

result = zero_shot(text, labels)
print(f'  Text: "{text[:60]}..."')
for label, score in zip(result['labels'], result['scores']):
    bar = '█' * int(score * 30)
    print(f'    {label:>15}: {bar} {score:.2%}')

# Production use case: route support tickets without training
print('\n--- Support Ticket Routing ---')
ticket = 'My card was charged twice for the same order and I want a refund.'
departments = ['billing', 'technical support', 'shipping', 'account management']
result = zero_shot(ticket, departments)
print(f'  Ticket: "{ticket[:60]}"')
print(f'  Route to: {result["labels"][0]} ({result["scores"][0]:.2%})')

---
## 3 · HuggingFace Datasets Deep Dive

### 🎓 Why This Matters

Before fine-tuning, you need data. The HuggingFace `datasets` library is the standard way
to load, process, and serve NLP datasets. Understanding it is **prerequisite knowledge** for
the LoRA fine-tuning workflows in the main curriculum (NB11).

### 3.1 · Two Modes: Map-style vs Streaming

```
  MAP-STYLE (Default)                    STREAMING (streaming=True)
  ┌──────────────────┐                   ┌──────────────────────────┐
  │ Downloads ENTIRE │                   │ Downloads ON DEMAND      │
  │ dataset to disk  │                   │ one batch at a time      │
  │                  │                   │                          │
  │ RAM: O(dataset)  │                   │ RAM: O(batch_size)       │
  │ Disk: Full copy  │                   │ Disk: ~0                 │
  │                  │                   │                          │
  │ Random access: ✓ │                   │ Random access: ✗         │
  │ dataset[42]      │                   │ Sequential iteration     │
  │                  │                   │                          │
  │ Use: < 50GB,     │                   │ Use: > 50GB, or you only │
  │ need shuffling   │                   │ need a sample            │
  └──────────────────┘                   └──────────────────────────┘
```

### 3.2 · When to Use Streaming

| Scenario | Map-style | Streaming |
|----------|-----------|----------|
| Dataset fits in RAM | ✅ Default | Overkill |
| Dataset > RAM (e.g., The Pile, 825GB) | ❌ OOM crash | ✅ Required |
| Quick exploration of huge dataset | ❌ Hours to download | ✅ Instant |
| Need random access / shuffling | ✅ Supported | ❌ Sequential only |
| Reproducible train/val split | ✅ `.train_test_split()` | ⚠️ Requires `.take()` / `.skip()` |

> **🧠 Socratic Question:** *The Pile dataset is 825GB. Your machine has 32GB RAM. How would*
> *you fine-tune an LLM on it? Streaming is the answer — you never hold more than one batch*
> *in memory. This is identical to how PyTorch's `IterableDataset` works.*

In [ ]:
from datasets import load_dataset

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 3.2a · Map-style loading (default): downloads to ~/.cache/huggingface/
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

print('=' * 65)
print('MAP-STYLE LOADING: load_dataset("imdb")')
print('=' * 65)

ds_map = load_dataset('imdb')

print(f'Type: {type(ds_map)}')
print(f'Splits: {list(ds_map.keys())}')
print(f'Train size: {len(ds_map["train"]):,} samples')
print(f'Test size:  {len(ds_map["test"]):,} samples')
print(f'Features: {ds_map["train"].features}')
print(f'\nRandom access demo: ds_map["train"][42]')
sample = ds_map['train'][42]
print(f'  Label: {sample["label"]} ({"positive" if sample["label"]==1 else "negative"})')
print(f'  Text:  "{sample["text"][:100]}..."')
print(f'\nMemory: the ENTIRE dataset is cached on disk and indexed in RAM.')

In [ ]:
from datasets import load_dataset

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# 3.2b · Streaming mode: NO download, iterate on-the-fly
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

print('=' * 65)
print('STREAMING MODE: load_dataset("imdb", streaming=True)')
print('=' * 65)

ds_stream = load_dataset('imdb', split='train', streaming=True)

print(f'Type: {type(ds_stream)}  (IterableDataset — no len()!)')
print(f'  → Cannot do ds_stream[42] — no random access.')
print(f'  → Can only iterate: for sample in ds_stream: ...')
print()

# .take(n) — grab only the first n samples (like LIMIT in SQL)
print('Streaming .take(5) demo — first 5 samples, no full download:')
print('-' * 65)
for i, sample in enumerate(ds_stream.take(5)):
    label_str = 'positive' if sample['label'] == 1 else 'negative'
    print(f'  Sample {i}: [{label_str:>8}] "{sample["text"][:70]}..."')

print()
print('Key operations on IterableDatasets:')
print('  .take(n)      → First n samples (for quick exploration)')
print('  .skip(n)      → Skip first n (manual train/val split)')
print('  .shuffle(seed, buffer_size=1000) → Approximate shuffle')
print('  .map(fn)      → Apply transformation (lazy, not materialized)')
print('  .filter(fn)   → Filter samples (lazy)')
print()
print('Production pattern for huge datasets:')
print('  train_stream = load_dataset("cerebras/SlimPajama-627B", streaming=True, split="train")')
print('  → 627B tokens, ~1TB on disk. Streaming = instant start, zero disk usage.')

---
## 4 · Domain-Adaptive Pretraining (DAPT) & Continuous Pre-training (CPT)

### 🎓 The Production Problem

Imagine you need to classify **highly technical medical literature** or **complex legal contracts**. You download `bert-base-uncased` (or a larger language model), but soon realize its vocabulary and internal representations are based on Wikipedia. It knows what a "cell" is (biological vs. prison), but it does not rigorously understand "myocardial infarction" or the strict syntax of an "indemnification clause".

If you only have 500 labeled medical/legal documents for fine-tuning, your model will struggle to learn the vocabulary and the task at the same time. How can we make the model an "expert" in our domain *before* we start fine-tuning on our small labeled dataset?

### The AI Engineering Decision Matrix
Before reaching for DAPT, evaluate your constraints. When do you use which technique?

| Objective | Technique | Data Required | Compute | Risk |
|-----------|-----------|---------------|---------|------|
| **Format Output** | Prompt Engineering | None | $0 | Hallucinates on complex jargon |
| **Add Knowledge** | RAG (Retrieval) | DB of docs | Low | Retriever might miss the context |
| **Teach Behavior / Tone** | Supervised Fine-Tuning (SFT) | 1k-10k labeled pairs | Medium | Doesn't embed deep new facts |
| **Teach New Language/Jargon** | **DAPT / CPT** | **1B+ unlabeled tokens** | **High** | **Catastrophic forgetting of base skills** |
| **Create Base Knowledge** | Pre-training from scratch | 1T+ unlabeled tokens | $1M+ | Financial ruin if botched |

DAPT/CPT is strictly for when a model fundamental "doesn't speak the jargon" and SFT isn't enough to teach it the semantics of the domain.


### Curriculum Learning Strategies
You cannot just dump 50GB of raw medical text into an LLM and run epochs. If you do, it will suffer from **Catastrophic Forgetting** — it will become great at medical text but "forget" how to be a helpful assistant or speak colloquially. 

**The Senior Approach (Curriculum Learning):**
1. **Phase 1 (General + Domain):** Mix 70% domain text with 30% generic replay data (Wikipedia/News) to preserve base capabilities.
2. **Phase 2 (Hard Domain):** Gradually increase the domain concentration to 90% in the final stages of the CPT run.
3. **Phase 3 (Task SFT):** Attach your classification head (or run instruction-tuning) on the now-adapted model.


In [ ]:
# Cell 4a — Simulating a Curriculum Learning Data Mix
from datasets import interleave_datasets, load_dataset

print("Loading unlabeled datasets for DAPT...")
# 1. Our highly specific domain data (e.g., medical abstracts)
# domain_ds = load_dataset('medical_text', streaming=True)

# 2. Generic replay data to prevent Catastrophic Forgetting
# generic_ds = load_dataset('wikipedia', '20220301.en', streaming=True)

# Create the Phase 1 Curriculum Mix (70% Medical, 30% Wikipedia)
# mixed_ds = interleave_datasets([domain_ds, generic_ds], probabilities=[0.7, 0.3], seed=42)
print("Curriculum created: 70% Domain / 30% Replay mixed stream.")


### The Full Continued Pre-training Pipeline
The code below demonstrates setting up a Masked Language Modeling (MLM) loop to force the model to learn the grammatical structure of the new domain unsupervised.


In [ ]:
# Cell 4b — DAPT Training Pipeline Setup
from transformers import AutoModelForMaskedLM, AutoTokenizer, DataCollatorForLanguageModeling, Trainer, TrainingArguments
import torch

model_id = 'bert-base-uncased'

# 1. Load base model with MLM Head (NOT a classification head)
tokenizer = AutoTokenizer.from_pretrained(model_id)
model_mlm = AutoModelForMaskedLM.from_pretrained(model_id)

# 2. Collator — This automatically masks 15% of tokens in each batch!
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=True,
    mlm_probability=0.15
)

# 3. DAPT-optimized Training Arguments
training_args_dapt = TrainingArguments(
    output_dir='./dapt_medical_checkpoint',
    per_device_train_batch_size=16,
    
    # DAPT CRITICAL RULE: Learning rate must be 10x-100x SMALLER than standard SFT
    # If LR is too high, it destroys the original pre-trained weights.
    learning_rate=2e-5,               
    warmup_ratio=0.05,
    # Even 1 epoch over a massive corpus is typically enough
    max_steps=5000,                   
    save_steps=1000,
    fp16=torch.cuda.is_available(), # AMP to save VRAM
)

# 4. Initialize Trainer
trainer_dapt = Trainer(
    model=model_mlm,
    args=training_args_dapt,
    # train_dataset=mixed_ds, # The 70/30 curriculum dataset
    data_collator=data_collator,
)

print('DAPT Pipeline Ready.')
print('After saving, you load this exact model ID using AutoModelForSequenceClassification to execute your final SFT phase.')


---
## 5 · Fine-tuning with Trainer API + Evaluation Metrics

### 🎓 Junior to Senior: The Gap in Most Tutorials

Most tutorials show `Trainer` without a `compute_metrics` function. The result:
you see loss going down but have **no idea** if the model actually classifies correctly.

**The Critical Insight:** Loss and metrics can diverge!

```
Epoch   Loss     F1      What's Happening
  1     0.69    0.50     Random guessing (loss = ln(2))
  2     0.45    0.72     Model is learning — both improve
  3     0.30    0.85     Healthy training
  4     0.15    0.84     ⚠️ Loss drops, F1 stagnates!
  5     0.05    0.82     🚨 OVERFITTING: loss keeps falling, F1 degrades
```

**Why does this happen?** Cross-entropy loss rewards **confident** predictions.
An overfit model learns to output `[0.99, 0.01]` on training data (low loss)
but the decision boundary on unseen data doesn't improve (flat/falling F1).
**Always track task-specific metrics alongside loss.**

### The Complete Pipeline

```
Pre-trained Model (knows language)  +  Your Data (100-10K labels)
         │                                        │
         └─────── Trainer API ────────────────────┘
                      │
              compute_metrics()
              (F1, Precision, Recall)
                      │
              Fine-tuned Model (knows your task)
```

This is the same `Trainer` used in NB11.  
For large models (7B+), use LoRA/QLoRA (NB11) instead of full fine-tuning.

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from datasets import load_dataset

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# STEP 1: Load a real dataset from the Hub and prepare train/eval splits
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

model_name = 'distilbert-base-uncased'
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Load IMDB — a real-world binary sentiment dataset (25K train, 25K test)
# We use a small subset for this demo to keep training fast.
full_dataset = load_dataset('imdb')

# Subsample for the demo: 500 train, 200 eval (enough to see real metrics)
train_dataset = full_dataset['train'].shuffle(seed=42).select(range(500))
eval_dataset  = full_dataset['test'].shuffle(seed=42).select(range(200))

print(f'Train samples: {len(train_dataset)}')
print(f'Eval samples:  {len(eval_dataset)}')
print(f'Labels: {full_dataset["train"].features["label"].names}')
print()

# ── Tokenize ─────────────────────────────────────────────────────────────────
def tokenize_fn(examples: dict) -> dict:
    """Tokenize text with truncation and padding for batch efficiency.
    
    Note: We use padding='max_length' here for simplicity. In production,
    use padding='longest' with a DataCollatorWithPadding for dynamic padding
    (pads each batch to its longest sequence, not the global max — saves compute).
    """
    return tokenizer(
        examples['text'],
        truncation=True,
        padding='max_length',
        max_length=256,
    )

train_tokenized = train_dataset.map(tokenize_fn, batched=True, desc='Tokenizing train')
eval_tokenized  = eval_dataset.map(tokenize_fn, batched=True, desc='Tokenizing eval')

# Remove the raw text column (Trainer expects only tensor-compatible columns)
train_tokenized = train_tokenized.remove_columns(['text'])
eval_tokenized  = eval_tokenized.remove_columns(['text'])

# Set format to PyTorch tensors
train_tokenized.set_format('torch')
eval_tokenized.set_format('torch')

print(f'Tokenized features: {list(train_tokenized.features.keys())}')
print(f'Input shape: {len(train_tokenized[0]["input_ids"])} tokens per sample')

In [ ]:
import numpy as np
import evaluate

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# STEP 2: Define compute_metrics — THE most important function students skip
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#
# The `evaluate` library provides standardized metric implementations.
# This function is called by Trainer at the end of each evaluation phase.

# Load metrics from the `evaluate` library
accuracy_metric  = evaluate.load('accuracy')
precision_metric = evaluate.load('precision')
recall_metric    = evaluate.load('recall')
f1_metric        = evaluate.load('f1')


def compute_metrics(eval_pred) -> dict:
    """Compute classification metrics from Trainer's EvalPrediction.

    Args:
        eval_pred: NamedTuple with:
            - predictions: np.ndarray of shape (n_samples, n_classes) — raw logits
            - label_ids:   np.ndarray of shape (n_samples,) — ground truth labels

    Returns:
        Dictionary of metric_name → float, logged by Trainer.

    Note:
        Trainer passes RAW LOGITS, not probabilities. We argmax to get predictions.
        For multi-class, use average='weighted'. For binary, 'binary' is fine.
    """
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    return {
        'accuracy':  accuracy_metric.compute(predictions=predictions, references=labels)['accuracy'],
        'f1':        f1_metric.compute(predictions=predictions, references=labels, average='binary')['f1'],
        'precision': precision_metric.compute(predictions=predictions, references=labels, average='binary')['precision'],
        'recall':    recall_metric.compute(predictions=predictions, references=labels, average='binary')['recall'],
    }


print('compute_metrics function defined.')
print('  Input:  EvalPrediction(logits, labels) from Trainer')
print('  Output: {accuracy, f1, precision, recall}')
print()
print('Why each metric matters:')
print('  • Accuracy:  Overall correctness (misleading if classes are imbalanced)')
print('  • Precision: Of predicted positives, how many are correct? (spam filter)')
print('  • Recall:    Of actual positives, how many did we find? (disease detection)')
print('  • F1:        Harmonic mean of Precision & Recall (balanced measure)')

In [ ]:
from transformers import TrainingArguments, Trainer
import torch

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# STEP 3: Configure Trainer with evaluation
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

training_args = TrainingArguments(
    output_dir='./sentiment_finetuned',
    
    # Training
    num_train_epochs=3,
    per_device_train_batch_size=16,
    learning_rate=2e-5,               # Standard for BERT fine-tuning
    weight_decay=0.01,                # L2 regularization
    warmup_ratio=0.1,                 # Warm up for 10% of steps
    
    # Evaluation — THIS IS WHAT MOST TUTORIALS MISS
    eval_strategy='epoch',            # Evaluate at end of each epoch
    per_device_eval_batch_size=32,    # Larger batch OK for eval (no gradients)
    
    # Logging
    logging_strategy='epoch',
    save_strategy='no',
    report_to='none',
    
    # Best model tracking
    load_best_model_at_end=False,     # Set True in production + save_strategy='epoch'
    seed=42,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=eval_tokenized,      # ← Evaluation dataset
    compute_metrics=compute_metrics,   # ← Our custom metrics function!
)

n_params = sum(p.numel() for p in model.parameters()) / 1e6
print(f'Model: {model_name} ({n_params:.0f}M parameters)')
print(f'Training: {len(train_tokenized)} samples, {training_args.num_train_epochs} epochs')
print(f'Evaluation: {len(eval_tokenized)} samples after each epoch')
print(f'Metrics tracked: accuracy, f1, precision, recall')
print()
print('Starting fine-tuning...')
print('Watch: loss should decrease AND F1 should increase.')
print('If loss drops but F1 stagnates → overfitting signal!')
print('─' * 65)

train_result = trainer.train()

print('─' * 65)
print(f'Training complete! Final loss: {train_result.training_loss:.4f}')

In [ ]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# STEP 4: Final evaluation and analysis
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

print('=' * 65)
print('FINAL EVALUATION ON HELD-OUT TEST SET')
print('=' * 65)

final_metrics = trainer.evaluate()

print(f'  Loss:      {final_metrics["eval_loss"]:.4f}')
print(f'  Accuracy:  {final_metrics["eval_accuracy"]:.4f}')
print(f'  F1:        {final_metrics["eval_f1"]:.4f}')
print(f'  Precision: {final_metrics["eval_precision"]:.4f}')
print(f'  Recall:    {final_metrics["eval_recall"]:.4f}')
print()

# ── Manual inference test ────────────────────────────────────────────────────
model.eval()
test_texts = [
    'This movie was absolutely brilliant, I loved every moment!',
    'Terrible film, completely boring and a waste of time.',
    'It was okay, had some good parts and some bad parts.',
]
inputs = tokenizer(test_texts, return_tensors='pt', padding=True, truncation=True, max_length=256)
# Move to same device as model
inputs = {k: v.to(model.device) for k, v in inputs.items()}

with torch.no_grad():
    logits = model(**inputs).logits
    probs = torch.softmax(logits, dim=-1)

label_names = ['negative', 'positive']
print('Manual inference test:')
print('─' * 65)
for text, prob in zip(test_texts, probs):
    pred_idx = prob.argmax().item()
    print(f'  [{label_names[pred_idx]:>8} {prob[pred_idx]:.2%}] "{text[:55]}..."')

### 🎓 Loss vs. F1: The Deep Explanation

**Why can loss decrease while F1 stagnates or drops?**

Cross-entropy loss and F1 measure **different things**:

| | Cross-Entropy Loss | F1 Score |
|--|-------------------|----------|
| **Measures** | Confidence calibration | Decision boundary quality |
| **Optimized by** | Gradient descent (differentiable) | Not differentiable — computed post-hoc |
| **Rewards** | Pushing probabilities toward 0 or 1 | Getting the argmax correct |
| **Failure mode** | Model becomes overconfident on training data | Model's boundary doesn't generalize |

**Concrete example:**
- Epoch 3: Model predicts `P(positive) = 0.7` for a positive sample → Loss = 0.36, Correct ✓
- Epoch 5: Model predicts `P(positive) = 0.99` for same sample → Loss = 0.01, Correct ✓
- The loss dropped **36x** but the F1 contribution is identical.
- Meanwhile, on eval data the model becomes overconfident on wrong predictions too.

**Production rule:** Use `load_best_model_at_end=True` with `metric_for_best_model='f1'` 
and `save_strategy='epoch'` to select the checkpoint with the best F1, not lowest loss.

---

### 🎓 Bridge to the Main Curriculum: What's Next?

You now have production-grade knowledge of:
- ✅ `datasets` library — loading, streaming, tokenizing
- ✅ `Trainer` API — training loops, evaluation strategy
- ✅ `evaluate` library — F1, Precision, Recall
- ✅ `compute_metrics` pattern

**This is exactly the foundation needed for NB11 (LoRA Fine-tuning):**

| What You Learned Here | How NB11 Extends It |
|----------------------|--------------------|
| Full fine-tuning (all 66M params) | LoRA: Only 0.1-1% of params are trainable |
| `Trainer(model=model)` | `Trainer(model=peft_model)` — same API! |
| `datasets` + tokenization | Same pipeline, bigger models (7B-70B) |
| `streaming=True` for big data | Essential for LLM training data (TB-scale) |
| `compute_metrics` with F1 | Same pattern, different metrics (perplexity for LLMs) |

> **Key insight:** The `Trainer` API is deliberately designed so that switching from
> full fine-tuning to LoRA requires changing **only the model**, not the data pipeline,
> training loop, or evaluation code. Master it once, use it everywhere.

### 5.3 · Overcoming the GPU VRAM Bottleneck

In production, you often want to use a **Batch Size of 32 or 64** to stabilize gradients and speed up training. However, even a 'base' model like BERT (110M params) with a batch size of 32 and a sequence length of 512 often exceeds **24GB of VRAM** (the limit of a consumer RTX 3090/4090).

When you see the dreaded `CUDA Out of Memory (OOM)` error, do not immediately lower your batch size to 1. Use these two essential software-based optimizations:

#### 1. Gradient Accumulation
If you can't fit a batch of 32 in memory, you can simulate it by running several smaller "micro-batches" and only updating the weights after a few steps.

- **The Math**: If you set `per_device_train_batch_size=8` and `gradient_accumulation_steps=4`, the model will:
  1. Run 8 samples, calculate gradients, but **NOT** update weights.
  2. Repeat this 4 times (summing the gradients in memory).
  3. On the 4th step, it runs the optimizer and resets the gradients.
- **The Result**: You get the mathematical stability of a **Batch Size of 32** while only ever holding **8 samples** in VRAM at once.

#### 2. Automatic Mixed Precision (AMP)
By default, PyTorch uses **FP32** (32-bit floats) for all math. **Mixed Precision** (`fp16` or `bf16`) stores activations and gradients in 16-bit format, which:
- Cuts VRAM requirements for activations **in half**.
- Speeds up training on modern GPUs with "Tensor Cores".
- **Safety**: The model maintains "master weights" in FP32 to prevent precision loss, converting them to 16-bit only for the intensive matrix multiplications.

#### Engineering Pattern: Optimized TrainingArguments
```python
training_args = TrainingArguments(
    output_dir="./optimized_finetuning",
    # ── VRAM Optimizations ──────────────────────────────────────────
    per_device_train_batch_size=8,   # Small micro-batch fits in VRAM
    gradient_accumulation_steps=4,   # 8 * 4 = virtual batch of 32
    fp16=True,                       # Use Mixed Precision (NVIDIA)
    # bf16=True,                     # Use BFloat16 (Newer NVIDIA/TPU)
    # ── Other params ────────────────────────────────────────────────
    learning_rate=2e-5,
    num_train_epochs=3,
)
```


---
### 5.1 · Advanced Technique: Handling Imbalanced Datasets

In production NLP (e.g., fraud detection, medical diagnosis, or toxicity filtering), classes are rarely balanced. If 99% of your data is 'Non-Toxic', a model can achieve 99% accuracy by simply predicting 'Non-Toxic' every time. 

To fix this, we must penalize the model more heavily when it misclassifies the minority class. We do this by applying **Class Weights** to the loss function.

**The Elite Pattern: Custom Trainer Subclass**
The HuggingFace `Trainer` is designed to be extensible. By overriding `compute_loss`, we can inject custom logic (like weighted cross-entropy) directly into the training loop.


In [ ]:
from torch import nn
from transformers import Trainer

class WeightedTrainer(Trainer):
    """Advanced Trainer that handles class imbalance using weighted loss."""
    
    def __init__(self, class_weights, *args, **kwargs):
        super().__init__(*args, **kwargs)
        # Move weights to the same device as the model
        self.class_weights = class_weights.to(self.model.device)

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        """Overriding the loss calculation to apply class weighting."""
        labels = inputs.get("labels")
        # Forward pass
        outputs = model(**inputs)
        logits = outputs.get("logits")
        
        # Define weighted CrossEntropyLoss
        loss_fct = nn.CrossEntropyLoss(weight=self.class_weights)
        loss = loss_fct(logits.view(-1, self.model.config.num_labels), labels.view(-1))
        
        return (loss, outputs) if return_outputs else loss

# --- Usage Example ---
# if class 0 has 1000 samples and class 1 has 100 samples:
# weights = torch.tensor([1.0, 10.0]) 
print("WeightedTrainer class defined. Use this instead of standard Trainer for imbalanced tasks.")

---
### 5.2 · Advanced Technique: Knowledge Distillation

**The Production Conflict:** We want the performance of a massive model (like `BERT-large` or `Llama-3`), but we need the latency and cost of a tiny model (like `DistilBERT`). 

**The Solution: Knowledge Distillation (Teacher-Student)**
Knowledge Distillation (Hinton et al., 2015) is the process of training a small model (the **Student**) to mimic the behavior of a large, pre-trained model (the **Teacher**).

#### How It Works:
1. **Soft Targets**: Instead of just learning from hard labels (0 or 1), the student tries to match the Teacher's full probability distribution (the **logits**).
2. **Temperature Scaling ($T$)**: We soften the Teacher's output distribution using a temperature parameter. Higher $T$ reveals the "dark knowledge" (the relationships between incorrect classes) that the model has learned.
3. **Loss Function**: The Student minimizes the **KL-Divergence** between its own predictions and the Teacher's soft targets.

**Why it works:** The Teacher's probabilities contain much more information than raw labels. For example, if the Teacher predicts a 0.001% chance of a cat image being a dog, but only 0.000001% chance of it being a car, it is signaling to the Student that dogs are more similar to cats than cars are. This structural knowledge allows the Student to achieve ~95% of the Teacher's performance at a fraction of the size.


---
### 5.4 · Advanced Technique: Token Classification & Label Alignment

#### The Production Nightmare: Misalignment
In **Token Classification** (e.g., Named Entity Recognition), each *word* in your training data has a single label (e.g., `B-LOC` for "Washington"). However, as we learned in NLP 01, Transformers use subword tokenization.

If "Washington" tokenizes into `["Wash", "##ing", "##ton"]`, how do we assign labels to these new tokens? If we simply duplicate `B-LOC` for all three, we're teaching the model that each piece is a separate entity start, which is factually wrong and ruins performance on long words.

#### The `-100` Rule
The industry-standard solution is to assign the label (`B-LOC`) to the **first subword only** and a special value of **`-100`** to all subsequent subwords.
1. **The Logic**: We only want the model to be penalized for identifying the entity itself.
2. **The `CrossEntropyLoss` Magic**: PyTorch’s loss functions are hardcoded to ignore the index `-100` during gradient calculation. By using this index, these subwords do not contribute to the loss or pollute the accuracy metrics.

#### Python Utility: Realigning Labels


In [ ]:
def align_labels_with_tokens(labels, word_ids):
    """
    Realigns word-level labels with subword tokens.
    
    Args:
        labels: List of word-level labels (e.g., [B-PER, O, B-LOC])
        word_ids: Mapping from token index to word index (from tokenizer)
    """
    new_labels = []
    current_word = None
    for word_id in word_ids:
        if word_id is None: # Special tokens ([CLS], [SEP])
            new_labels.append(-100)
        elif word_id != current_word: # Start of a new word
            new_labels.append(labels[word_id])
            current_word = word_id
        else: # Continuation of a word (subword)
            new_labels.append(-100)
    return new_labels

# Example: "Washington" (B-LOC) -> ["Wash", "##ing", "##ton"]
word_ids = [None, 0, 0, 0, None] # CLS, Wash, ing, ton, SEP
word_labels = [3] # B-LOC is ID 3
aligned = align_labels_with_tokens(word_labels, word_ids)
print(f'Aligned Labels: {aligned}') # [-100, 3, -100, -100, -100]

---
### 5.5 · Handling Long Documents: The Sliding Window Strategy

#### 1. The 512-Token Hard Ceiling
A common production scenario: You need to classify a **2,000-word legal document**, but BERT's architecture has a fixed maximum sequence length of **512 tokens**.
- **The Junior Approach**: Truncate the text (`truncation=True`). You lose 75% of the information, potentially missing critical clauses at the end of the document.
- **The Senior Approach**: Implement a **Sliding Window (Chunking)** strategy.

#### 2. The Algorithm: Chunk, Infer, Pool
Instead of feeding the whole document at once, we break it into overlapping segments:
1. **Divide**: Split the document into chunks of 512 tokens with an **overlap** (e.g., 128 tokens) to preserve context at the boundaries.
2. **Batch Inference**: Pass each chunk through the model independently.
3. **Logit Aggregation**: Take the `[CLS]` output logits for all chunks and perform **Mean Pooling** or **Max Pooling** to get a single unified prediction for the entire document.

**Conceptual Implementation:**
```python
# Use the tokenizer's built-in sliding window support
inputs = tokenizer(
    long_text,
    max_length=512,
    stride=128,              # The overlap between windows
    return_overflowing_tokens=True,
    truncation=True,
    return_tensors='pt'
)

# inputs['input_ids'] will now have shape [NUM_CHUNKS, 512]
with torch.no_grad():
    logits = model(**inputs).logits # [NUM_CHUNKS, NUM_LABELS]

# Aggregation: Mean pooling across chunks
final_logits = torch.mean(logits, dim=0)
prediction = torch.argmax(final_logits)
```


---
### 5.6 · Strategic Fine-Tuning: Layer Freezing & Discriminative Learning Rates

#### 1. The Catastrophic Forgetting Risk
If you fine-tune the entire BERT model (110M+ parameters) on a **tiny dataset** (e.g., 500 medical records), the model will "over-adjust." It may memorize the rare terminology in your dataset while **forgetting** the general linguistic nuances it learned during its 100B token pre-training.

#### 2. Strategy A: Selective Layer Freezing
To preserve the model's fundamental language skills, we can freeze the **Embeddings** and the **Lower Encoder Layers** (which capture general syntax) and only train the **Upper Layers** and the **Classification Head**.

```python
# Freeze Embeddings and first 8 layers
for param in model.bert.embeddings.parameters():
    param.requires_grad = False

for layer in model.bert.encoder.layer[:8]:
    for param in layer.parameters():
        param.requires_grad = False
```

#### 3. Strategy B: Discriminative Fine-Tuning (ULMFiT)
Instead of a single learning rate for the whole model, we use an **exponentially decaying** learning rate across layers:
- **Classification Head**: High LR ($2e^{-5}$) — needs to learn the new task.
- **Top Layers**: Medium LR ($1e^{-5}$).
- **Bottom Layers**: Low LR ($1e^{-6}$) — only needs minimal tweaks.

**The Result**: The model effectively "specializes" its high-level semantic layers while keeping its foundational syntactic layers stable, drastically improving performance on small-data regimes.


---
## 6 · Production Deployment Patterns

| Pattern | Latency | Cost | Best For |
|---------|---------|------|----------|
| **API (OpenAI, HF Inference)** | 200-500ms | Pay per request | Prototyping, low volume |
| **Self-hosted (FastAPI + GPU)** | 50-100ms | Fixed GPU cost | Medium volume, custom models |
| **Optimized (ONNX/TensorRT)** | 5-20ms | Optimized GPU | High volume, latency-critical |
| **Edge (ONNX Runtime)** | 10-50ms | No GPU needed | On-device, privacy-sensitive |

### Connection to the Main Curriculum

| This Notebook | Main Track |
|--------------|------------|
| Pipeline basics | → NB15 (FastAPI serving) |
| Fine-tuning with Trainer | → NB11 (LoRA fine-tuning) |
| Zero-shot classification | → NB24 (Prompt engineering) |
| NER pipeline | → NB28 (Entity extraction for RAG) |
| Model selection | → NB11 (Model selection helper) |

### 🎓 DEEP QUESTION ANSWERED

**Q:** *How do you choose between fine-tuning a base model vs. using an existing fine-tuned model?*

**A:** Decision tree:
1. Is there an existing model fine-tuned for your EXACT task? → Try it first (free)
2. Does it meet your accuracy bar (>90% on your test set)? → Deploy it
3. If not, is your domain very specialized (legal, medical, finance)? → Fine-tune yourself
4. Do you have enough labeled data (>500 examples)? → Fine-tune
5. Too little data? → Use zero-shot classification or few-shot prompting (NB24)

**Risk of using existing models:** They may have been trained on data that doesn't match your distribution. A sentiment model trained on movie reviews may fail on medical patient feedback.

**Risk of fine-tuning yourself:** Overfitting on small datasets, catastrophic forgetting of pre-trained knowledge, and ongoing maintenance burden.

---

### 6.1 · Optimized Inference: ONNX Export & Dynamic Quantization

#### The Production Reality
Deploying a raw PyTorch Transformer for high-volume inference is often too slow and expensive for CPU-only environments. To achieve ultra-low latency, we use **ONNX (Open Neural Network Exchange)** and **Quantization**.
1. **ONNX Export**: Converts the model into a static computation graph optimized by the ONNX Runtime (ORT).
2. **Dynamic Quantization (int8)**: Reduces weight precision from 32-bit floats (FP32) to 8-bit integers (int8). This cuts model size by 4x and can result in **3x faster CPU inference** with negligible accuracy loss.

#### Elite Pattern: Using HuggingFace `optimum`

In [ ]:
# Note: Requires 'pip install optimum[onnxruntime]'
# from optimum.onnxruntime import ORTModelForSequenceClassification
# from transformers import AutoTokenizer, pipeline

print("Deployment Pattern Snippet (Optimizing 0.1s → 0.03s latency):")
print("# 1. Export and Quantize in one command")
print('model = ORTModelForSequenceClassification.from_pretrained("sentiment_finetuned", export=True, quantization_config="avx512")')
print()
print("# 2. Use in a standard Pipeline")
print('quantized_pipeline = pipeline("sentiment-analysis", model=model, tokenizer=tokenizer)')


## ✅ Knowledge Check

### Q1: Pipeline internals
What three steps does `pipeline('sentiment-analysis')(text)` perform internally?

<details><summary>👉 Answer</summary>

1. **Tokenization**: Convert text to token IDs using the model's tokenizer
2. **Model inference**: Feed token IDs through the model to get logits
3. **Post-processing**: Apply softmax to logits, map to labels (POSITIVE/NEGATIVE), return structured output
</details>

### Q2: Learning rate
Why is the fine-tuning learning rate (2e-5) SO much lower than training from scratch (~1e-3)?

<details><summary>👉 Answer</summary>

The pre-trained weights already encode useful language knowledge. A high learning rate would destroy these learned features (catastrophic forgetting). A low learning rate makes small, careful adjustments to adapt the existing knowledge to the new task.
</details>

### Q3: Zero-shot vs fine-tuning
You have 50 labeled examples and need 95% accuracy. Should you fine-tune or use zero-shot?

<details><summary>👉 Answer</summary>

Start with zero-shot classification — 50 examples is often too few for reliable fine-tuning. If zero-shot doesn't hit 95%, try few-shot prompting with an LLM (NB24). If that doesn't work, invest in labeling more data (aim for 500+) before fine-tuning. 50 examples should be your TEST set, not training set.
</details>

### Q4: Streaming datasets
You need to fine-tune a model on a 200GB text dataset. Your machine has 16GB RAM. What do you do?

<details><summary>👉 Answer</summary>

Use `load_dataset(name, streaming=True)`. This returns an `IterableDataset` that downloads data on-the-fly, one batch at a time. RAM usage is O(batch_size), not O(dataset_size). The `Trainer` supports `IterableDataset` natively. You lose random access and must use `.shuffle(buffer_size=N)` for approximate shuffling.
</details>

### Q5: Loss vs F1
Your training loss dropped from 0.5 to 0.05 over 10 epochs, but eval F1 peaked at epoch 4 and has been declining. What happened and what should you do?

<details><summary>👉 Answer</summary>

Classic overfitting. After epoch 4, the model is memorizing training data (loss drops) but the decision boundary on unseen data is degrading (F1 drops). Solutions: (1) Use `load_best_model_at_end=True` with `metric_for_best_model='f1'` to keep the epoch-4 checkpoint. (2) Add regularization (higher `weight_decay`, `dropout`). (3) Reduce epochs. (4) Add more training data.
</details>

---

## 🔨 Practical Practice

### Exercise 1: NER Pipeline for RAG
Build a pipeline that extracts named entities from a document, then creates metadata tags for each entity. This is preprocessing for RAG (NB28).

### Exercise 2: Model Comparison
Compare 3 sentiment models on the same 20 test sentences: `distilbert-base-uncased-finetuned-sst-2-english`, `nlptown/bert-base-multilingual-uncased-sentiment`, and a zero-shot classifier. Which is most accurate? Which is fastest?

### Exercise 3: Streaming + Metrics
Load a large dataset (e.g., `amazon_polarity`) with `streaming=True`. Take 1000 samples with `.take()`, fine-tune for 1 epoch, and evaluate with `compute_metrics`. Compare the F1 vs loading the full dataset.

---

## 🎉 NLP Track Complete!

You've covered the full evolution:

```
NLP 01: Text → Tokens          (how computers read)
NLP 02: Tokens → Vectors        (how meaning becomes geometry)
NLP 03: Vectors → Sequences     (how RNNs process language)
NLP 04: Sequences → Attention   (how context is captured)
NLP 05: Attention → Pre-training (how models learn language)
NLP 06: Pre-training → Production (how to ship it)
```

**Next →** `16_07_model_development.ipynb` — Model Development: LoRA, QLoRA & Supervised Fine-Tuning